# 🔬 Eksperimen Gabungan 1 & 2: Cross-Project Flaky Test Prediction (Full Factorial 72 Pipelines)
**Kelompok Peneliti**: Kelompok A01 — RPL A (Magister Teknik Informatika ITS)  
**Tujuan Eksperimen**: Menyatukan dan membandingkan secara komprehensif **Tri-Paradigma Data Cleaning (C1: Row Drop vs C2: Project Drop vs C3: Imputasi Cerdas)** yang dikombinasikan dengan 3 Scaling, 4 Imbalance, dan 2 Classifiers dalam matriks **72 Faktorial Pipelines**.  
**Skema Validasi**: Leave-One-Project-Out Cross-Validation (LOPO-CV, Zero Data Leakage).

## 1. Verifikasi Lingkungan & Hardware GPU RTX 4050

In [ ]:
import sys
import os
from pathlib import Path

src_dir = str(Path("../src").resolve())
if src_dir not in sys.path:
    sys.path.append(src_dir)

import check_gpu
check_gpu.check_nvidia_smi()
check_gpu.check_python_environment()
check_gpu.check_xgboost_gpu()

## 2. Ingestion & Komparasi Awal Tri-Paradigma Data (C1, C2, C3)

In [ ]:
from data_loader import load_dataset

df_c1, feat_c1, proj_c1 = load_dataset('C1')
df_c2, feat_c2, proj_c2 = load_dataset('C2')
df_c3, feat_c3, proj_c3 = load_dataset('C3')

print(f"\n[KOMPARASI AWAL DATASET]:")
print(f"  C1 (Row-Level Drop)     : {len(df_c1):,} baris | {len(proj_c1)} proyek | {df_c1['flaky'].sum()} flaky ({df_c1['flaky'].mean()*100:.2f}%)")
print(f"  C2 (Project-Level Drop) : {len(df_c2):,} baris | {len(proj_c2)} proyek | {df_c2['flaky'].sum()} flaky ({df_c2['flaky'].mean()*100:.2f}%)")
print(f"  C3 (Imputasi Cerdas)    : {len(df_c3):,} baris | {len(proj_c3)} proyek | {df_c3['flaky'].sum()} flaky ({df_c3['flaky'].mean()*100:.2f}%)")

## 3. Eksekusi Matriks 72 Pipeline LOPO-CV (Single-Run)

In [ ]:
from lopo_runner import run_all_72_pipelines

# Eksekusi seluruh 72 konfigurasi faktorial (C1: 24 folds, C2: 19 folds, C3: 24 folds)
df_summary = run_all_72_pipelines(output_dir="../results", use_gpu=True)
display(df_summary.head(10))

## 4. Uji Signifikansi Statistik (Friedman & Nemenyi CD)

In [ ]:
from statistical_tests import run_friedman_nemenyi_test, run_cleaning_comparison_test

run_friedman_nemenyi_test("../results/summary_metrics_72_pipelines.csv")
run_cleaning_comparison_test("../results/summary_metrics_72_pipelines.csv")

## 5. Visualisasi Publikasi Ilmiah

In [ ]:
from visualizer import generate_all_visualizations

generate_all_visualizations(summary_csv_path="../results/summary_metrics_72_pipelines.csv", plots_dir="../results/plots")
print("Visualisasi tersimpan di results/plots/")